# Multi-Agent Research Coordinator Lab

A guided notebook for the hub-and-spoke research workflow implemented in [`agent.py`](./agent.py). The notebook inspects and exercises that implementation rather than maintaining a second copy of the orchestration logic.

**Learning path:** identify the hub and spokes → inspect SDK/tool configuration → follow explicit context passing → verify gap refinement offline → optionally run live research.

No API calls are made unless you explicitly enable the final live-run cell.

## 1. Load the lab package

Run this cell first. It locates the repository root so the notebook works whether the kernel starts in the project root or this folder.

In [ ]:
from pathlib import Path
import sys

here = Path.cwd().resolve()
ROOT = next(
    (path for path in (here, *here.parents) if (path / "multi_agent_research" / "agent.py").is_file()),
    None,
)
if ROOT is None:
    raise FileNotFoundError("Could not find multi_agent_research/agent.py from the current directory")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from multi_agent_research import agent

print(f"Repository root: {ROOT}")
print(f"Coordinator function: {agent.run_research.__name__}")

## 2. Identify coordinator and subagents

`run_research()` is the **coordinator (hub)**. It owns the sequence and state. The **Web Search** and **Document Analysis** specialists are the two spokes. Planning, coverage evaluation, and report synthesis are additional coordinator turns with their own prompts and structured output models.

The coordinator and each specialist return structured data: for example, `ResearchPlan`, `WebFindings`, `DocumentAnalysis`, and `CoverageReview`. These types make handoffs inspectable and testable.

In [ ]:
roles = {
    "Coordinator / planner": (agent.PLANNER_PROMPT, agent.ResearchPlan),
    "Web Search spoke": (agent.WEB_PROMPT, agent.WebFindings),
    "Document Analysis spoke": (agent.DOCUMENT_PROMPT, agent.DocumentAnalysis),
    "Coverage evaluator": (agent.COVERAGE_PROMPT, agent.CoverageReview),
    "Final synthesis": (agent.SYNTHESIS_PROMPT, agent.ResearchReport),
}
for name, (instructions, output_type) in roles.items():
    print(f"{name}: {output_type.__name__}")
    print(instructions.strip().splitlines()[0])
    print()

## 3. See how a role becomes an SDK configuration

A role name does not itself create tools. `_sdk_call()` maps a role to its system prompt and `allowed_tools`, puts those settings into `ClaudeAgentOptions`, and calls the SDK's `query(prompt=..., options=...)`. The Web Search specialist gets `WebSearch` and `WebFetch`; Document Analysis gets no tools and can only use context passed in its prompt.

The next cell prints the actual adapter source so you can trace the code in sequence.

In [ ]:
import inspect

print(inspect.getsource(agent._sdk_call))

### What to notice

1. The `role` selects the specialist instructions and tool allowlist.
2. `ClaudeAgentOptions` configures model, system prompt, tools, turn limit, and structured output schema.
3. `query()` runs the model. When the model requests an enabled built-in tool, the SDK/runtime handles it and returns the result to the model.
4. The adapter checks the final SDK result and validates it against the requested Pydantic model.

Tool availability depends on the Claude Agent SDK runtime. Prompt text alone cannot enable a tool.

## 4. Inspect the renewable-energy coverage checks

This lab sets an explicit benchmark for renewable-energy topics. Fusion is included because the exercise requests it, but the final synthesis must distinguish it from currently deployed renewable energy sources.

In [ ]:
topic = "Renewable energy technologies"
required_topics = agent._energy_requirements(topic)
print("Minimum planned subtopics:", agent.MIN_SUBTOPICS)
print("Required renewable-energy coverage:", required_topics)

## 5. Follow explicit context passing

The hub sends the plan to Web Search. Its `WebFindings` output is then passed along with the plan and any user-supplied document text to Document Analysis. In the first round, the call looks like this conceptually:

In [ ]:
import json

example_assignment = {
    "id": "tidal",
    "title": "Tidal energy",
    "objectives": ["Compare tidal-range and tidal-stream power, maturity, benefits, and limits."],
}
example_web_context = {
    "topic": topic,
    "assignments": [example_assignment],
}
example_document_context = {
    "topic": topic,
    "assignments": [example_assignment],
    "user_supplied_documents": ["Optional document excerpt supplied by the user."],
    "web_research_excerpts": {
        "findings": ["Example finding to illustrate the handoff; not real research."],
        "sources": [{"title": "Example only", "url": "https://example.test", "note": "Placeholder"}],
        "limitations": ["This is illustrative placeholder data."],
    },
}
print("Web specialist receives:")
print(json.dumps(example_web_context, indent=2))
print("\nDocument specialist receives:")
print(json.dumps(example_document_context, indent=2))

The example above is just a visualization. In the real workflow, `run_research()` serializes the current plan and actual findings for every specialist call. The coverage evaluator receives the plan plus both specialists' outputs. If some subtopic is weak or missing, the coordinator sends that gap, previous findings, and the evaluator's rationale back to the spokes.

## 6. Verify the orchestration offline

Run the focused test. It injects scripted specialists instead of calling the live SDK. The test deliberately identifies tidal and fusion as first-pass gaps, checks they are delegated again, and verifies final coverage of solar, wind, geothermal, tidal, biomass, and fusion. It also checks that documents and web findings reach Document Analysis.

This tests orchestration and context passing, not the factual accuracy of live web results.

In [ ]:
import subprocess

test_file = ROOT / "multi_agent_research" / "tests" / "test_research_coordinator.py"
completed = subprocess.run(
    [sys.executable, "-m", "pytest", str(test_file), "-q"],
    cwd=ROOT,
    text=True,
    capture_output=True,
)
print(completed.stdout)
if completed.returncode:
    print(completed.stderr)
    raise RuntimeError(f"Offline coordinator tests failed with exit code {completed.returncode}")

## 7. Optional live research run

**Safe by default:** leave `RUN_LIVE = False` to avoid any model/API or web calls. To run live, set `ANTHROPIC_API_KEY`, confirm your SDK runtime has access to `WebSearch` and `WebFetch`, then change the flag to `True` and execute the cell. Live search can take time and may incur usage costs.

In [ ]:
RUN_LIVE = False  # Change to True only when you intend to make a live SDK call.

if RUN_LIVE:
    import asyncio
    import os

    if not os.environ.get("ANTHROPIC_API_KEY"):
        raise RuntimeError("Set ANTHROPIC_API_KEY before enabling the live run.")
    live_report = await agent.run_research(topic)
    display(live_report)
    print("Coverage complete:", live_report.coverage_complete)
    print("Coverage gaps:", live_report.coverage_gaps)

## 8. Your turn

Try a different broad subject. Before running live, think about:

1. What five or more subtopics would make a balanced plan?
2. What source or document context should each spoke receive?
3. What evidence would count as adequate coverage?
4. What gap would trigger another delegation round?

For a custom topic, set `topic` in an earlier cell and use the optional live cell. The scripted renewable-energy test remains the deterministic check for the required six technologies.